In [1]:
#1.Load kant.txt using the Colab file manager
#2.Downloading the file from GitHub
!curl -L https://raw.githubusercontent.com/Denis2054/Transformers-for-NLP-and-Computer-Vision-3rd-Edition/master/Chapter06/kant.txt --output "kant.txt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10.7M  100 10.7M    0     0  11.8M      0 --:--:-- --:--:-- --:--:-- 11.8M


In [2]:
!pip install Transformers
!pip install --upgrade accelerate
from accelerate import Accelerator

In [3]:
from pathlib import Path
from tokenizers import ByteLevelBPETokenizer

paths = [str(x) for x in Path(".").glob("**/*.txt")]

# Read the content from the files, ignoring or replacing invalid characters
file_contents = []
for path in paths:
  try:
    with open(path, 'r', encoding='utf-8', errors='replace') as file:
      file_contents.append(file.read())
  except Exception as e:
    print(f"Error reading {path}: {e}")

# Join the contents into a single string
text = "\n".join(file_contents)

# Initialize a tokenizer
tokenizer = ByteLevelBPETokenizer()

# Customize training
tokenizer.train_from_iterator([text], vocab_size=52000, min_frequency=2,
                              special_tokens=[
                                  "<s>",
                                  "<pad>",
                                  "</s>",
                                  "<unk>",
                                  "<mask>"
                              ])

In [4]:
import os

token_dir = '/content/KantaiBERT'
if not os.path.exists(token_dir):
  os.makedirs(token_dir)
tokenizer.save_model('KantaiBERT')

['KantaiBERT/vocab.json', 'KantaiBERT/merges.txt']

In [5]:
from tokenizers.implementations import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing

tokenizer = ByteLevelBPETokenizer(
    "./KantaiBERT/vocab.json",
    "./KantaiBERT/merges.txt"
)

In [6]:
tokenizer.encode("The Critique of Pure Reason.").tokens

['The', 'ĠCritique', 'Ġof', 'ĠPure', 'ĠReason', '.']

In [7]:
tokenizer.encode("The Critique of Pure Reason.")

Encoding(num_tokens=6, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [8]:
tokenizer._tokenizer.post_processor = BertProcessing(
    ("</s>", tokenizer.token_to_id("</s>")),
    ("<s>", tokenizer.token_to_id("<s>"))
)
tokenizer.enable_truncation(max_length=512)

In [9]:
tokenizer.encode("The Critique of Pure Reason.")

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [10]:
tokenizer.encode("The Critique of Pure Reason.").tokens

['<s>', 'The', 'ĠCritique', 'Ġof', 'ĠPure', 'ĠReason', '.', '</s>']

In [11]:
!nvidia-smi

Thu May  7 02:41:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
import torch

torch.cuda.is_available()

True

In [13]:
from transformers import RobertaConfig

config = RobertaConfig(
    vocab_size=52000,
    max_position_embeddings=514,
    num_attention_heads=12,
    num_hidden_layers=6,
    type_vocab_size=1
)

In [14]:
from transformers import RobertaTokenizer
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained("./KantaiBERT", max_length=512)

In [15]:
from transformers import RobertaForMaskedLM

In [16]:
model = RobertaForMaskedLM(config=config)

In [17]:
print(model)

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): La

In [18]:
print(model.num_parameters())

83504416


In [19]:
LP = list(model.parameters())
lp = len(LP)
print(lp)

106


In [20]:
for p in range(0, lp):
  print(LP[p])

Parameter containing:
tensor([[-0.0034,  0.0011,  0.0270,  ...,  0.0325,  0.0061, -0.0223],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0010,  0.0037, -0.0334,  ..., -0.0223,  0.0111,  0.0078],
        ...,
        [-0.0217, -0.0042, -0.0159,  ..., -0.0086,  0.0142,  0.0142],
        [ 0.0096,  0.0142, -0.0411,  ...,  0.0220, -0.0134,  0.0129],
        [-0.0194, -0.0071,  0.0238,  ..., -0.0105, -0.0145,  0.0226]],
       requires_grad=True)
Parameter containing:
tensor([[-5.8422e-03,  9.6485e-03, -7.4391e-03, -3.2397e-03,  3.6192e-02,
          9.6361e-03,  4.8704e-02,  3.0719e-02,  7.7112e-03, -3.1236e-03,
         -1.5534e-02,  2.0488e-02, -3.7440e-02, -3.1510e-02,  2.0136e-02,
          6.2045e-03, -2.3290e-02,  3.3072e-02,  3.5266e-03,  1.6222e-02,
         -1.8856e-02,  1.0996e-02,  5.8773e-03, -5.1313e-03, -4.0883e-02,
          2.3000e-02,  1.6882e-02, -1.3270e-02, -7.7937e-03,  1.2268e-02,
         -4.9495e-02,  1.1098e-02,  4.5211e-02, -1.

In [21]:
# Shape of each tensor in the model
LP = list(model.parameters())
for i, tensor in enumerate(LP):
  print(f"Shape of tensor {i}: {tensor.shape}")

Shape of tensor 0: torch.Size([52000, 768])
Shape of tensor 1: torch.Size([1, 768])
Shape of tensor 2: torch.Size([768])
Shape of tensor 3: torch.Size([768])
Shape of tensor 4: torch.Size([514, 768])
Shape of tensor 5: torch.Size([768, 768])
Shape of tensor 6: torch.Size([768])
Shape of tensor 7: torch.Size([768, 768])
Shape of tensor 8: torch.Size([768])
Shape of tensor 9: torch.Size([768, 768])
Shape of tensor 10: torch.Size([768])
Shape of tensor 11: torch.Size([768, 768])
Shape of tensor 12: torch.Size([768])
Shape of tensor 13: torch.Size([768])
Shape of tensor 14: torch.Size([768])
Shape of tensor 15: torch.Size([3072, 768])
Shape of tensor 16: torch.Size([3072])
Shape of tensor 17: torch.Size([768, 3072])
Shape of tensor 18: torch.Size([768])
Shape of tensor 19: torch.Size([768])
Shape of tensor 20: torch.Size([768])
Shape of tensor 21: torch.Size([768, 768])
Shape of tensor 22: torch.Size([768])
Shape of tensor 23: torch.Size([768, 768])
Shape of tensor 24: torch.Size([768])
Sh

In [22]:
#counting the parameters
np = 0
for p in range(0, lp):  # number of tensors
  PL2 = True
  try:
    L2 = len(LP[p][0])  # check if 2D
  except:
    L2 = 1              #  not 2D but 1D
    PL2 = False
  L1 = len(LP[p])
  L3 = L1 * L2
  np += L3              # number of parameters per tensor
  if PL2 == True:
    print(p, L1, L2, L3)  # displaying the sizes of the parameters
  if PL2 == False:
    print(p, L1, L3)    # displaying the sizes of the parameters

print(np)

0 52000 768 39936000
1 1 768 768
2 768 768
3 768 768
4 514 768 394752
5 768 768 589824
6 768 768
7 768 768 589824
8 768 768
9 768 768 589824
10 768 768
11 768 768 589824
12 768 768
13 768 768
14 768 768
15 3072 768 2359296
16 3072 3072
17 768 3072 2359296
18 768 768
19 768 768
20 768 768
21 768 768 589824
22 768 768
23 768 768 589824
24 768 768
25 768 768 589824
26 768 768
27 768 768 589824
28 768 768
29 768 768
30 768 768
31 3072 768 2359296
32 3072 3072
33 768 3072 2359296
34 768 768
35 768 768
36 768 768
37 768 768 589824
38 768 768
39 768 768 589824
40 768 768
41 768 768 589824
42 768 768
43 768 768 589824
44 768 768
45 768 768
46 768 768
47 3072 768 2359296
48 3072 3072
49 768 3072 2359296
50 768 768
51 768 768
52 768 768
53 768 768 589824
54 768 768
55 768 768 589824
56 768 768
57 768 768 589824
58 768 768
59 768 768 589824
60 768 768
61 768 768
62 768 768
63 3072 768 2359296
64 3072 3072
65 768 3072 2359296
66 768 768
67 768 768
68 768 768
69 768 768 589824
70 768 768
71 768 768

In [33]:
%%time
from datasets import load_dataset
from transformers import DataCollatorForLanguageModeling

# 1. Load the raw text file
raw_datasets = load_dataset("text", data_files={"train": "./kant.txt"})

# 2. Tokenize the data
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

dataset = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# 3. Data Collator handles the padding/masking
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

CPU times: user 38.2 ms, sys: 3.01 ms, total: 41.2 ms
Wall time: 286 ms


In [35]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./KantaiBERT",
    num_train_epochs=1,
    per_device_train_batch_size=64,
    save_steps=10000,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset["train"]
)

In [36]:
%%time
trainer.train()

Step,Training Loss
500,6.627043
1000,5.763842
1500,5.310308
2000,5.039961
2500,4.862389


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CPU times: user 10min 19s, sys: 3.02 s, total: 10min 23s
Wall time: 11min 21s


TrainOutput(global_step=2942, training_loss=5.413926405294018, metrics={'train_runtime': 680.616, 'train_samples_per_second': 276.642, 'train_steps_per_second': 4.323, 'total_flos': 958259296373184.0, 'train_loss': 5.413926405294018, 'epoch': 1.0})

In [37]:
trainer.save_model("./KantaiBERT")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [41]:
from transformers import pipeline

fill_mask = pipeline(
    "fill-mask",
    model="./KantaiBERT",
    tokenizer="./KantaiBERT"
)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

In [42]:
fill_mask("Human thinking involves human <mask>.")

[{'score': 0.010119847021996975,
  'token': 393,
  'token_str': ' reason',
  'sequence': 'Human thinking involves human  reason.'},
 {'score': 0.005154421087354422,
  'token': 601,
  'token_str': ' understanding',
  'sequence': 'Human thinking involves human  understanding.'},
 {'score': 0.005053152795881033,
  'token': 532,
  'token_str': ' experience',
  'sequence': 'Human thinking involves human  experience.'},
 {'score': 0.004628945142030716,
  'token': 606,
  'token_str': ' conceptions',
  'sequence': 'Human thinking involves human  conceptions.'},
 {'score': 0.003958306275308132,
  'token': 791,
  'token_str': ' proposition',
  'sequence': 'Human thinking involves human  proposition.'}]